# Ensemble Learning: Bagging, Boosting, and Stacking

## Lesson Objectives
By the end of this lesson, students should be able to:
- Explain **bagging** and **boosting** intuitively and mathematically at a high level
- Describe **Random Forest (RF)**, **XGBoost**, and **LightGBM**
- Understand **model averaging** and **stacking (stacked generalization)**
- Build regression models on a **real housing dataset**
- Compare models using **MAE, RMSE, and R²**
- Perform **hyperparameter tuning** using cross-validation
- Implement **model averaging and stacking** in practice


## 1. Bagging vs Boosting vs Stacking

### Bagging (Bootstrap Aggregating)
- Train multiple models on different sampled datasets
- Combine predictions (average for regression)



### Boosting
- Models are trained **sequentially**
- Each model corrects previous errors

### Stacking (Stacked Generalization)
- Train multiple base models
- Use their predictions as inputs to a **meta-model**

### Model Averaging
- Simple version of stacking
- Combine predictions using mean (or weighted mean)


## 2. Key Models

### Random Forest (Bagging)
- Ensemble of decision trees
- Random feature selection at each split

### XGBoost (Boosting)
- Gradient boosting with regularization

### LightGBM (Boosting)
- Faster boosting using histogram-based splits


In [1]:
# Install if needed
# !pip install xgboost lightgbm

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import LinearRegression

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


In [2]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

print("Shape:", df.shape)
df.head()

Shape: (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [3]:
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
def evaluate(name, model):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    return {"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2}, preds

## How Boosting Works

Boosting builds models sequentially, where each new model focuses on correcting previous errors.

### Key Idea:
Instead of independent models, models depend on each other.

### Steps:
1. Train a first model
2. Measure its errors
3. Train the next model to focus more on those errors
4. Repeat the process
5. Combine all models into a final prediction

### Why it works:
The model gradually learns complex patterns by fixing mistakes step-by-step

### Intuition:
"Each new model says: what did the last model get wrong?"



## How Bagging Works

Bagging (Bootstrap Aggregating) improves model stability by reducing variance.

### Key Idea:
Instead of training one model, we train many models on different samples of the data.

### Steps:
1. Draw multiple bootstrap samples (sampling with replacement) from the training data
2. Train a separate model (e.g., decision tree) on each sample
3. Combine predictions:
   - Regression → average
   - Classification → majority vote

### Why it works:
Each model sees slightly different data → errors cancel out when averaged

### Intuition:
"Many weak, noisy models → one stable and reliable model"

## How Random Forest Works

Random Forest is a bagging-based ensemble of decision trees.

### Key Mechanisms:
- Bootstrap sampling (different data per tree)
- Random feature selection at each split

### Prediction:
- Regression → average of all tree outputs

### Why randomness helps:
- Reduces correlation between trees
- Improves generalization

### Intuition:
"Many diverse trees → strong combined prediction"

## How LightGBM Works

LightGBM is a fast gradient boosting framework.

### Key Mechanisms:
- Uses histogram-based splitting (faster)
- Grows trees leaf-wise (more efficient learning)

### Advantages:
- Faster training
- Handles large datasets well

### Risk:
- Can overfit if not tuned properly

### Intuition:
"A faster and more efficient version of boosting"

## How Model Averaging Works

Model averaging combines predictions from multiple models.

### Formula:
Final Prediction = (Model1 + Model2 + Model3) / 3

### Why it works:
- Different models capture different patterns
- Averaging reduces individual model errors

### Limitation:
- All models are treated equally (no learning of importance)

### Intuition:
"Take the average opinion of multiple models"

## How Stacking Works

Stacking combines multiple models using a meta-model.

### Key Idea:
Instead of averaging, we **learn how to combine models**

### Steps:
1. Train base models (RF, XGB, LGBM)
2. Use their predictions as new features
3. Train a meta-model (e.g., Linear Regression)
4. Meta-model learns optimal combination

### Why it works:
- Learns which model performs better in different situations

### Intuition:
"A model learns how to combine other models intelligently"

In [5]:
rf = RandomForestRegressor(n_estimators=200, random_state=42)
xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
lgbm = LGBMRegressor(n_estimators=300, learning_rate=0.05, random_state=42)

rf_res, rf_pred = evaluate("RF", rf)
xgb_res, xgb_pred = evaluate("XGB", xgb)
lgbm_res, lgbm_pred = evaluate("LGBM", lgbm)

pd.DataFrame([rf_res, xgb_res, lgbm_res])

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001684 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1838
[LightGBM] [Info] Number of data points in the train set: 16512, number of used features: 8
[LightGBM] [Info] Start training from score 2.071947


,Model,MAE,RMSE,R2
0,RF,0.326812,0.503960,0.806186
1,XGB,0.306975,0.466968,0.833595
2,LGBM,0.296576,0.449810,0.845599


## 3. Model Averaging (Ensemble)

We simply average predictions from all models.


In [6]:
avg_pred = (rf_pred + xgb_pred + lgbm_pred) / 3

mae = mean_absolute_error(y_test, avg_pred)
rmse = np.sqrt(mean_squared_error(y_test, avg_pred))
r2 = r2_score(y_test, avg_pred)

{"Model": "Averaging", "MAE": mae, "RMSE": rmse, "R2": r2}

{'Model': 'Averaging',
 'MAE': 0.30138427245761046,
 'RMSE': np.float64(0.46184474593298225),
 'R2': 0.8372259554935003}

## 4. Stacking (Advanced Ensemble)

We use a **meta-model (Linear Regression)** to combine predictions.


In [7]:
stack_model = StackingRegressor(
    estimators=[
        ('rf', rf),
        ('xgb', xgb),
        ('lgbm', lgbm)
    ],
    final_estimator=LinearRegression()
)

stack_res, stack_pred = evaluate("Stacking", stack_model)
stack_res

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000679 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1838
[LightGBM] [Info] Number of data points in the train set: 16512, number of used features: 8
[LightGBM] [Info] Start training from score 2.071947
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000330 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1838
[LightGBM] [Info] Number of data points in the train set: 13209, number of used features: 8
[LightGBM] [Info] Start training from score 2.067432
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000335 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1837
[LightGBM] [Info] Number of data points in the train set: 13209, number of used features: 8
[LightGBM] [Info] Start traini

{'Model': 'Stacking',
 'MAE': 0.29676239248536357,
 'RMSE': np.float64(0.45138199357379516),
 'R2': 0.8445174693505512}

## 5. Hyperparameter Tuning (Random Forest Example)

We use **GridSearchCV** with cross-validation.


In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10],
    "min_samples_split": [2, 5]
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)

best_rf = grid.best_estimator_
best_res, _ = evaluate("Tuned RF", best_rf)
best_res

In [ ]:
results = pd.DataFrame([rf_res, xgb_res, lgbm_res, stack_res, best_res])
results.sort_values(by="RMSE")

## Exercises

1. Change test size to 30% and compare results  
2. Tune XGBoost hyperparameters  
3. Tune LightGBM hyperparameters  
4. Try weighted averaging instead of simple average  
5. Change stacking meta-model to RandomForest  
6. Add a new model (e.g., GradientBoostingRegressor)  
7. Compare training time vs accuracy  

---

### Advanced Challenge
- Build a **custom stacking pipeline manually**
- Compare stacking vs averaging vs single best model
